In [1]:
# Cell 1 - Environment check
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'WARNING: No GPU detected — this will be slow on CPU')


Sat Jun 20 01:29:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.20             Driver Version: 570.133.20     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:61:00.0 Off |                    0 |
| N/A   34C    P0             50W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Cell 2 - Configuration
import torch, torch.nn as nn, torch.nn.functional as F
import math, json, time, os
import numpy as np
from pathlib import Path
from dataclasses import dataclass, asdict

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

PROJECT_ID = 'attnonly_std_seed_completion'
OUT_DIR = Path('/workspace') / PROJECT_ID if Path('/workspace').exists() else Path('.') / PROJECT_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model hyperparameters — IDENTICAL to original 10M ResidualGPT run ──
N_LAYER    = 6
N_EMBD     = 384
N_HEAD     = 6
BLOCK_SIZE = 256
DROPOUT    = 0.2

# ── Training hyperparameters ──────────────────────────────────────────
BATCH_SIZE      = 64
GRAD_ACCUM      = 1
MAX_ITERS       = 3000
EVAL_INTERVAL   = 300
EVAL_BATCHES    = 50
LEARNING_RATE   = 3e-4
WEIGHT_DECAY    = 0.1
BETA1, BETA2    = 0.9, 0.95
GRAD_CLIP       = 1.0

# ── THE ONLY EXPERIMENTS THIS NOTEBOOK RUNS ──────────────────────────
# seed 1337 is a sanity check ONLY (see Cell 9), not a full retrain.
SANITY_SEED = 1337
NEW_SEEDS   = [42, 123]

# Reference value from the already-confirmed seed-1337 run — used to
# validate this reimplementation produces the same architecture/behavior.
REFERENCE_SEED1337_BEST_VAL = 1.579988667964935

print(f'Project: {PROJECT_ID}')
print(f'New seeds to run: {NEW_SEEDS}')
print(f'Reference (seed 1337, already confirmed): {REFERENCE_SEED1337_BEST_VAL:.6f}')


Device: cuda
Project: attnonly_std_seed_completion
New seeds to run: [42, 123]
Reference (seed 1337, already confirmed): 1.579989


In [3]:
# Cell 3 - Dataset (TinyShakespeare, character-level — identical to original)
import urllib.request

DATA_DIR = OUT_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)
txt_path = DATA_DIR / 'input.txt'

if not txt_path.exists():
    print('Downloading TinyShakespeare...')
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        txt_path
    )
text = txt_path.read_text()

chars = sorted(set(text))
VOCAB_SIZE = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f'Vocab size: {VOCAB_SIZE}')
print(f'Train tokens: {len(train_data):,}')
print(f'Val tokens:   {len(val_data):,}')


Vocab size: 65
Train tokens: 1,003,854
Val tokens:   111,540


In [4]:
# Cell 4 - Batcher with FIXED validation batches (critical architecture detail)
#
# This is the single most important fix from the original nanoGPT pilot:
# random val batches at every eval introduce noise that masks slow learners
# like AttnOnly. Fixed batches (seed 99,991) ensure identical evaluation
# data across every step and every seed, so seed-to-seed variance reflects
# the model, not the evaluation sample.

class SequenceBatcher:
    def __init__(self, train_data, val_data, block_size, batch_size, device,
                 fixed_val_seed=99991, n_fixed_val_batches=50):
        self.train_data = train_data
        self.val_data   = val_data
        self.block_size = block_size
        self.batch_size = batch_size
        self.device     = device

        # Build fixed val batches ONCE, using an isolated generator so it
        # never interacts with the global / per-experiment seed state.
        g = torch.Generator().manual_seed(fixed_val_seed)
        self.fixed_val_batches = []
        for _ in range(n_fixed_val_batches):
            ix = torch.randint(len(val_data) - block_size, (batch_size,), generator=g)
            xb = torch.stack([val_data[i:i+block_size]     for i in ix]).to(device)
            yb = torch.stack([val_data[i+1:i+block_size+1] for i in ix]).to(device)
            self.fixed_val_batches.append((xb, yb))

    def sample(self, split):
        d = self.train_data if split == 'train' else self.val_data
        ix = torch.randint(len(d) - self.block_size, (self.batch_size,))
        xb = torch.stack([d[i:i+self.block_size]     for i in ix]).to(self.device)
        yb = torch.stack([d[i+1:i+self.block_size+1] for i in ix]).to(self.device)
        return xb, yb

# Build ONCE — shared across all seeds in this notebook, exactly as in
# the original ResidualGPT run, so every config sees identical val data.
BATCHER = SequenceBatcher(train_data, val_data, BLOCK_SIZE, BATCH_SIZE, DEVICE)
print(f'Fixed val batches built: {len(BATCHER.fixed_val_batches)} batches, seed=99991')


Fixed val batches built: 50 batches, seed=99991


In [5]:
# Cell 5 - ResidualGPT model
# Critical correctness requirements baked in here:
#   (a) NO runtime gain multiplier in forward() — gain is applied to WEIGHTS
#       at init time only, via apply_output_gains(), never as a forward-pass
#       multiplier. (That bug was caught and fixed in the original project —
#       see paper Section 5.1.)
#   (b) Weight tying between token_embed and lm_head.
#   (c) attention_skip / mlp_skip flags control residual connections.

@dataclass
class ResidualSwitch:
    use_attention_skip: bool = True
    use_mlp_skip:       bool = True
    attention_gain:     float = 1.0
    mlp_gain:           float = 1.0


class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        assert N_EMBD % N_HEAD == 0
        self.qkv  = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
        self.out  = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.drop = nn.Dropout(DROPOUT)
        self.n_head = N_HEAD
        self.head_dim = N_EMBD // N_HEAD

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.drop.p if self.training else 0.0, is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.out(y))


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(N_EMBD, 4 * N_EMBD, bias=False)
        self.fc2  = nn.Linear(4 * N_EMBD, N_EMBD, bias=False)
        self.drop = nn.Dropout(DROPOUT)

    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))


class Block(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch    = switch
        self.ln1       = nn.LayerNorm(N_EMBD)
        self.ln2       = nn.LayerNorm(N_EMBD)
        self.attn      = CausalSelfAttention()
        self.mlp       = MLP()

    def forward(self, x):
        # NO runtime multiplier here. Gain is purely a weight-init-time effect.
        a = self.attn(self.ln1(x))
        x = x + a if self.switch.use_attention_skip else a
        m = self.mlp(self.ln2(x))
        x = x + m if self.switch.use_mlp_skip else m
        return x


class ResidualGPT(nn.Module):
    def __init__(self, switch: ResidualSwitch):
        super().__init__()
        self.switch = switch
        self.token_embed = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_embed   = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop        = nn.Dropout(DROPOUT)
        self.blocks      = nn.ModuleList([Block(switch) for _ in range(N_LAYER)])
        self.ln_f        = nn.LayerNorm(N_EMBD)
        self.lm_head     = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)

        # Weight tying
        self.lm_head.weight = self.token_embed.weight

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def apply_output_gains(self):
        """Weight-level gain application, called ONCE before optimizer construction.
        With gain=1.0 (this run's config) this is a no-op, included only for
        architectural parity with the original implementation."""
        if self.switch.mlp_gain != 1.0:
            with torch.no_grad():
                for block in self.blocks:
                    block.mlp.fc2.weight.mul_(self.switch.mlp_gain)
        if self.switch.attention_gain != 1.0:
            with torch.no_grad():
                for block in self.blocks:
                    block.attn.out.weight.mul_(self.switch.attention_gain)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.token_embed(idx) + self.pos_embed(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def param_count(self):
        return sum(p.numel() for p in self.parameters())

print('Model class defined')


Model class defined


In [6]:
# Cell 6 - Training and evaluation helpers

def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)

def learning_rate_at(step, max_iters, base_lr, min_lr=3e-5, warmup_iters=100):
    # EXACT match to Untitled__9_.ipynb Cell 12 learning_rate_at():
    #   - Linear warmup over warmup_iters (NOT half-cosine)
    #   - Cosine decay floors at min_lr (NOT 0)
    #   - decay_iters == max_iters here, so step>=max_iters -> min_lr
    decay_iters = max_iters
    if step < warmup_iters:
        return base_lr * (step + 1) / max(1, warmup_iters)
    if step >= decay_iters:
        return min_lr
    progress = (step - warmup_iters) / max(1, decay_iters - warmup_iters)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_lr + cosine * (base_lr - min_lr)

@torch.no_grad()
def estimate_val_loss(model, batcher, n_batches):
    model.eval()
    losses = []
    for xb, yb in batcher.fixed_val_batches[:n_batches]:
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

@torch.no_grad()
def estimate_train_loss(model, batcher, n_batches):
    model.eval()
    losses = []
    for _ in range(n_batches):
        xb, yb = batcher.sample('train')
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

def make_optimizer(model):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (decay if p.ndim >= 2 else no_decay).append(p)
    return torch.optim.AdamW(
        [{'params': decay, 'weight_decay': WEIGHT_DECAY},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=LEARNING_RATE, betas=(BETA1, BETA2)
    )

def train_one_run(seed, switch, max_iters=MAX_ITERS, eval_interval=EVAL_INTERVAL,
                   eval_batches=EVAL_BATCHES, verbose=True):
    set_seed(seed)
    model = ResidualGPT(switch).to(DEVICE)
    model.apply_output_gains()  # weight-level only, before optimizer
    optimizer = make_optimizer(model)

    history = []
    best_val = float('inf')
    best_step = -1
    started = time.time()

    # step 0 eval
    val0 = estimate_val_loss(model, BATCHER, eval_batches)
    train0 = estimate_train_loss(model, BATCHER, eval_batches)
    history.append({'step': 0, 'train_loss': train0, 'val_loss': val0})
    if val0 < best_val:
        best_val, best_step = val0, 0

    for step in range(1, max_iters + 1):
        lr = learning_rate_at(step, max_iters, LEARNING_RATE)
        for g in optimizer.param_groups:
            g['lr'] = lr

        optimizer.zero_grad(set_to_none=True)
        xb, yb = BATCHER.sample('train')
        _, loss = model(xb, yb)
        loss.backward()
        if GRAD_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        if step % eval_interval == 0 or step == max_iters:
            val_loss = estimate_val_loss(model, BATCHER, eval_batches)
            train_loss = estimate_train_loss(model, BATCHER, eval_batches)
            is_best = val_loss < best_val
            if is_best:
                best_val, best_step = val_loss, step
            history.append({'step': step, 'train_loss': train_loss, 'val_loss': val_loss})
            if verbose:
                elapsed = time.time() - started
                print(f'  seed={seed} step={step:5d} train={train_loss:.4f} '
                      f'val={val_loss:.4f} best={best_val:.4f}@{best_step} '
                      f'({elapsed:.0f}s)')

    return {
        'seed': seed,
        'best_step': best_step,
        'best_val_loss': best_val,
        'best_val_ppl': math.exp(min(best_val, 20)),
        'final_step': max_iters,
        'final_train_loss': history[-1]['train_loss'],
        'final_val_loss': history[-1]['val_loss'],
        'final_val_ppl': math.exp(min(history[-1]['val_loss'], 20)),
        'history': history,
        'param_count': model.param_count(),
    }

print('Training functions ready')


Training functions ready


In [7]:
# Cell 7 - Resume / checkpoint safety (not required for this short run,
# included for parity with the main project notebooks)
CKPT_DIR = OUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

def save_result(name, result):
    path = CKPT_DIR / f'{name}.json'
    with open(path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'Saved: {path}')

def load_result_if_exists(name):
    path = CKPT_DIR / f'{name}.json'
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

print('Checkpoint helpers ready')


Checkpoint helpers ready


In [8]:
# Cell 8 - The exact AttnOnly configuration under test
ATTN_ONLY_SWITCH = ResidualSwitch(
    use_attention_skip=True,
    use_mlp_skip=False,
    attention_gain=1.0,
    mlp_gain=1.0,
)
print('AttnOnly_std config:', ATTN_ONLY_SWITCH)


AttnOnly_std config: ResidualSwitch(use_attention_skip=True, use_mlp_skip=False, attention_gain=1.0, mlp_gain=1.0)


In [9]:
# Cell 9 - SANITY CHECK (run this BEFORE trusting seeds 42/123)
#
# Reruns seed 1337 for only 300 steps (10% of full run) and compares the
# trajectory shape against expectations. This is NOT meant to reproduce
# the exact final number (300 steps != 3000 steps) — it is meant to catch
# gross architecture mismatches (e.g. if val loss is flat/diverging when
# it should be descending, something differs from the original setup).
#
# Original full run reference (3000 steps): best_val_loss = 1.579988668

print('Running sanity check: seed=1337, 300 steps only...')
print(f'(Full run reference @ 3000 steps: {REFERENCE_SEED1337_BEST_VAL:.6f})')
print()

sanity_result = train_one_run(
    seed=SANITY_SEED,
    switch=ATTN_ONLY_SWITCH,
    max_iters=300,
    eval_interval=100,
    eval_batches=EVAL_BATCHES,
    verbose=True,
)

print()
print(f'Sanity check result @ step 300: val_loss = {sanity_result["final_val_loss"]:.4f}')
print()
print('PASS CRITERIA: val_loss should be descending and roughly in the')
print('1.6-2.0 range by step 300 (the original AttnOnly_std curve drops')
print('from ~4.3 at step 0 to ~1.85 by step 1000, ~1.58 by step 3000).')
print('If val_loss is still > 3.0 or increasing at step 300, STOP —')
print('something in this reimplementation differs from the original.')


Running sanity check: seed=1337, 300 steps only...
(Full run reference @ 3000 steps: 1.579989)

  seed=1337 step=  100 train=2.5336 val=2.5281 best=2.5281@100 (13s)
  seed=1337 step=  200 train=2.4312 val=2.4369 best=2.4369@200 (23s)
  seed=1337 step=  300 train=2.3906 val=2.4043 best=2.4043@300 (33s)

Sanity check result @ step 300: val_loss = 2.4043

PASS CRITERIA: val_loss should be descending and roughly in the
1.6-2.0 range by step 300 (the original AttnOnly_std curve drops
from ~4.3 at step 0 to ~1.85 by step 1000, ~1.58 by step 3000).
If val_loss is still > 3.0 or increasing at step 300, STOP —
something in this reimplementation differs from the original.


In [10]:
# Cell 10 - THE EXPERIMENT: AttnOnly_std at seeds 42 and 123
#
# Only run this cell after confirming Cell 9's sanity check looks correct.

results = {}

for seed in NEW_SEEDS:
    name = f'AttnOnly_std_seed{seed}'
    existing = load_result_if_exists(name)
    if existing is not None:
        print(f'Skipping {name} — already completed (best_val={existing["best_val_loss"]:.4f})')
        results[name] = existing
        continue

    print(f'\n{"="*70}')
    print(f'Running: {name}  (full {MAX_ITERS} steps)')
    print(f'{"="*70}')

    result = train_one_run(seed=seed, switch=ATTN_ONLY_SWITCH)
    result['name'] = name
    result['family'] = 'attnonly_threshold'
    result['attention_skip'] = True
    result['mlp_skip'] = False
    result['attention_gain'] = 1.0
    result['mlp_gain'] = 1.0

    save_result(name, result)
    results[name] = result

    print(f'\n{name} complete: best_val_loss = {result["best_val_loss"]:.6f} '
          f'@ step {result["best_step"]}')

print('\nAll requested seeds complete.')



Running: AttnOnly_std_seed42  (full 3000 steps)
  seed=42 step=  300 train=2.3098 val=2.3317 best=2.3317@300 (28s)
  seed=42 step=  600 train=1.9648 val=2.0681 best=2.0681@600 (54s)
  seed=42 step=  900 train=1.7413 val=1.9287 best=1.9287@900 (80s)
  seed=42 step= 1200 train=1.6064 val=1.8059 best=1.8059@1200 (106s)
  seed=42 step= 1500 train=1.5156 val=1.7243 best=1.7243@1500 (132s)
  seed=42 step= 1800 train=1.4551 val=1.6815 best=1.6815@1800 (158s)
  seed=42 step= 2100 train=1.4155 val=1.6456 best=1.6456@2100 (184s)
  seed=42 step= 2400 train=1.3907 val=1.6266 best=1.6266@2400 (210s)
  seed=42 step= 2700 train=1.3720 val=1.6108 best=1.6108@2700 (236s)
  seed=42 step= 3000 train=1.3614 val=1.6040 best=1.6040@3000 (261s)
Saved: /workspace/attnonly_std_seed_completion/checkpoints/AttnOnly_std_seed42.json

AttnOnly_std_seed42 complete: best_val_loss = 1.604021 @ step 3000

Running: AttnOnly_std_seed123  (full 3000 steps)
  seed=123 step=  300 train=2.2938 val=2.3213 best=2.3213@300 (28

In [11]:
# Cell 11 - Combine all 3 seeds and compute mean/std for the paper

all_three_seeds = {
    1337: REFERENCE_SEED1337_BEST_VAL,  # already confirmed, not rerun here
    42:   results.get('AttnOnly_std_seed42', {}).get('best_val_loss'),
    123:  results.get('AttnOnly_std_seed123', {}).get('best_val_loss'),
}

print('AttnOnly_std — full 3-seed result:')
print('=' * 50)
vals = []
for seed, val in all_three_seeds.items():
    status = 'CONFIRMED (existing)' if seed == 1337 else 'NEW (this run)'
    if val is not None:
        print(f'  seed={seed:<6} best_val_loss={val:.6f}  [{status}]')
        vals.append(val)
    else:
        print(f'  seed={seed:<6} MISSING — rerun Cell 10')

if len(vals) == 3:
    mean_val = float(np.mean(vals))
    std_val  = float(np.std(vals))
    print()
    print(f'Mean ± std across 3 seeds: {mean_val:.6f} ± {std_val:.6f}')
    print()
    print('Compare against the paper claim: 1.5725 ± 0.0060')
    print(f'This run produces:              {mean_val:.4f} ± {std_val:.4f}')

    final_summary = {
        'config': 'AttnOnly_std',
        'seeds': list(all_three_seeds.keys()),
        'best_val_per_seed': all_three_seeds,
        'mean_best_val_loss': mean_val,
        'std_best_val_loss': std_val,
    }
    with open(OUT_DIR / 'attnonly_std_3seed_final.json', 'w') as f:
        json.dump(final_summary, f, indent=2)
    print(f'\nSaved final 3-seed summary to {OUT_DIR / "attnonly_std_3seed_final.json"}')
else:
    print('\nIncomplete — rerun Cell 10 to fill in missing seeds before trusting mean/std.')


AttnOnly_std — full 3-seed result:
  seed=1337   best_val_loss=1.579989  [CONFIRMED (existing)]
  seed=42     best_val_loss=1.604021  [NEW (this run)]
  seed=123    best_val_loss=1.615744  [NEW (this run)]

Mean ± std across 3 seeds: 1.599918 ± 0.014883

Compare against the paper claim: 1.5725 ± 0.0060
This run produces:              1.5999 ± 0.0149

Saved final 3-seed summary to /workspace/attnonly_std_seed_completion/attnonly_std_3seed_final.json
